In [1]:
import torch
import numpy as np
import torchvision

In [2]:
# 1. Initialization
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch: 2.13.0+cu130
CUDA available: True


In [3]:
# 2. Review

print("Some tensor initialization")

tensor1 = torch.ones(2,3)                               # (2,3)
tensor2 = torch.rand_like(tensor1, dtype=torch.float)   # (2,3)
tensor3 = tensor2                                       # (2,3)
tensor4 = tensor1 @ tensor2.T                           # (2,2)
tensor5 = torch.empty(0)                                # (0)

print(tensor1)
print(tensor2)
print(tensor3)
print(tensor4)
print(tensor5)

Some tensor initialization
tensor([[1., 1., 1.],
        [1., 1., 1.]])
tensor([[0.1286, 0.6430, 0.3422],
        [0.5287, 0.7617, 0.5531]])
tensor([[0.1286, 0.6430, 0.3422],
        [0.5287, 0.7617, 0.5531]])
tensor([[1.1138, 1.8435],
        [1.1138, 1.8435]])
tensor([])


In [4]:
print("Some slicing")

print(tensor2[0,0])
print(tensor2[0,:])
print(tensor2[:,1])
print(tensor2[...,-1])
print(tensor2[0:2,0:2])

Some slicing
tensor(0.1286)
tensor([0.1286, 0.6430, 0.3422])
tensor([0.6430, 0.7617])
tensor([0.3422, 0.5531])
tensor([[0.1286, 0.6430],
        [0.5287, 0.7617]])


In [5]:
# 3. Shape operations

# Interpret this as: 2 batches x 3 rows x 4 features 
x = torch.arange(24).reshape(2, 3, 4)
print(f"{x}\n")

y = torch.empty(0)

# Tensor x[0]
print("Predicted dimension: (3,4)")
y = x[0]
print(y)
print(f"Actual dimension: {y.shape}\n")

# Tensor x[:, 1]
print("Predicted dimension: (2,1,4)")
y = x[:, 1]
print(y)
print(f"Actual dimension: {y.shape}\n")

# Tensor x[:, :, -1]
print("Predicted dimension: (2,3,1)") # I guess dimension with size 1 is always truncated
y = x[:, :, -1]
print(y)
print(f"Actual dimension: {y.shape}\n")

tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])

Predicted dimension: (3,4)
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
Actual dimension: torch.Size([3, 4])

Predicted dimension: (2,1,4)
tensor([[ 4,  5,  6,  7],
        [16, 17, 18, 19]])
Actual dimension: torch.Size([2, 4])

Predicted dimension: (2,3,1)
tensor([[ 3,  7, 11],
        [15, 19, 23]])
Actual dimension: torch.Size([2, 3])



In [6]:
# Tensor x[1, :, 1:3]
print("Predicted dimension: (3,2)")
y = x[1, :, 1:3]
print(y)
print(f"Actual dimension: {y.shape}\n")

# Tensor x[..., 0]
print("Predicted dimension: (2,3)")
y = x[..., 0]
print(y)
print(f"Actual dimension: {y.shape}\n")

Predicted dimension: (3,2)
tensor([[13, 14],
        [17, 18],
        [21, 22]])
Actual dimension: torch.Size([3, 2])

Predicted dimension: (2,3)
tensor([[ 0,  4,  8],
        [12, 16, 20]])
Actual dimension: torch.Size([2, 3])



In [7]:
# 4. Reshape & flatten

# Reshape can either return a view, or a copy of a tensor, depending on the context

y = x.reshape(6,4)
print(torch.equal(y.flatten(),x.flatten())) # This is how you do element-wise equivalence check
print(y.numel() == 24)                      # torch.eq(a,b) instead returns a boolean tensor (set bit indicates equal element)
print("\n")

y = x.reshape(2,12)
print(torch.equal(y.flatten(),x.flatten()))
print(y.numel() == 24)
print("\n")

y = x.reshape(-1,8) # Pytorch inferred the "-1" dimension based on size of the rest of dimensions to fit sum 24
print(y)
print(torch.equal(y.flatten(),x.flatten()))
print(y.numel() == 24)
print("\n")


True
True


True
True


tensor([[ 0,  1,  2,  3,  4,  5,  6,  7],
        [ 8,  9, 10, 11, 12, 13, 14, 15],
        [16, 17, 18, 19, 20, 21, 22, 23]])
True
True




In [8]:
# 5. unsqueeze and squeeze

v = torch.arange(4)

# unsqueeze(dim): return a new tensor with dimension of size one inserted at position dim.
v1 = v.unsqueeze(0)
print(v1.shape)

v2 = v.unsqueeze(1)
print(v2.shape)

v3 = v.unsqueeze(0).unsqueeze(2) # unsqueeze() can only insert one dimension at a time
print(v3.shape)

# squeeze(dim): returns a new tensor, with dimension of size one collapsed at specified position(s) dim
# (collapse all if dim is empty, though refrain as it can accidentally delete batch dimensions)

v1 = v1.squeeze(0)
v2 = v2.squeeze(1)
v3 = v3.squeeze()
print(f"{v1.shape} {v2.shape} {v3.shape}")

torch.Size([1, 4])
torch.Size([4, 1])
torch.Size([1, 4, 1])
torch.Size([4]) torch.Size([4]) torch.Size([4])


In [9]:
# transpose & permute

images = torch.randn(8, 3, 24, 32) # Return a tensor filled with random numbers with normal dist, mean 0, variance 1

# intepret axes as (N,C,H,W)

# permute(input, dims): permute dimensions of input according to dims ordering
# transpose(input, dim0, dim1): return transposed input, with dim0 and dim1 swapped

# (N,C,W,H)
images1 = images.transpose(2,3)
print(images1.shape)

# (N,H,W,C)
images2 = images1.permute((0,3,2,1))
print(images2.shape)

# (N,C,H,W)
images3 = images2.permute((0,3,1,2))
print(images3.shape)
print(images3.equal(images))

torch.Size([8, 3, 32, 24])
torch.Size([8, 24, 32, 3])
torch.Size([8, 3, 24, 32])
True


In [10]:
# Boolean mask indexing & integer array indexing

scores = torch.tensor(
[
    [4.0, -2.0, 7.0],
    [-1.0, 5.0, 0.0],
    [8.0, 3.0, -4.0],
    [2.0, 9.0, 1.0],
])

# Boolean mask: Applying a condition onto a tensor will cause that conditioned to be performend element-wise
# Result is a "boolean mask" where each set bit represent an element satisfying the condition

mask = scores > 0
print(mask)

# We can perform indexing with a boolean mask on tensors

positive_values = scores[mask]
print("Expected shape: (8)")
print(positive_values)

modified_scores = scores.clone()
modified_scores[(mask == 0)] = 0.0
print(modified_scores)

# Integer array indexing

row_indices = torch.tensor([2,0,3], dtype=torch.long)
selected_rows = scores[row_indices]
print(selected_rows)

print(scores)


tensor([[ True, False,  True],
        [False,  True, False],
        [ True,  True, False],
        [ True,  True,  True]])
Expected shape: (8)
tensor([4., 7., 5., 8., 3., 2., 9., 1.])
tensor([[4., 0., 7.],
        [0., 5., 0.],
        [8., 3., 0.],
        [2., 9., 1.]])
tensor([[ 8.,  3., -4.],
        [ 4., -2.,  7.],
        [ 2.,  9.,  1.]])
tensor([[ 4., -2.,  7.],
        [-1.,  5.,  0.],
        [ 8.,  3., -4.],
        [ 2.,  9.,  1.]])


In [11]:
"""

Simple Vectorized Pairwise Squared Distance

Given

points_a.shape == (N, D)
points_b.shape == (M, D)

We are expected to compute

distance_{i,j} = \\sum_{k=0}^{D-1}(points_a{i,k} - points_b{j,k})^{2}

"""

torch.manual_seed(7)
points_a = torch.randn(4, 3)
points_b = torch.randn(5, 3)

print("Expected output shape: (4,5)")

def pairwise_squared_distances_slow(
    points_a: torch.Tensor,
    points_b: torch.Tensor,
) -> torch.Tensor:
    distance = torch.zeros(points_a.shape[0],points_b.shape[0], dtype=torch.float)
    for i in range (0, points_a.shape[0]):
        for j in range (0, points_b.shape[0]):
            for k in range (0, points_a.shape[1]):
                distance[i,j] += (points_a[i,k] - points_b[j,k]) ** 2
    return distance

res1 = pairwise_squared_distances_slow(points_a, points_b)

assert torch.all(res1 >= 0)
assert res1.shape == (4,5)

res2 = pairwise_squared_distances_slow(points_a, points_a)
print(res1)
print(res2)

Expected output shape: (4,5)
tensor([[ 0.5797,  5.5144,  8.9752,  4.9353,  5.9582],
        [ 8.9153,  7.8217, 19.9707,  4.6841, 20.2838],
        [ 2.8075,  4.3572, 11.1966,  3.9171, 10.4436],
        [ 4.9109, 13.6777, 22.7070,  1.9204, 13.3243]])
tensor([[0.0000, 5.1462, 0.8967, 3.2395],
        [5.1462, 0.0000, 1.8529, 3.2639],
        [0.8967, 1.8529, 0.0000, 2.5965],
        [3.2395, 3.2639, 2.5965, 0.0000]])


In [12]:
"""

Broadcasting is a way to perform an operation between tensors that have similarities in their shapes.

Rules for broadcasting are:
- Each tensor must have at least one dimension - no empty tensors
- Comparing dimension sizes of the two tensors, going from *last to first*:
    - Each dimension must be **equal**, or
    - One of the dimension must be **size of 1**, or
    - The dimension **does not exist** in one of the tensor

"""

print(points_a)
print(points_b)

def pairwise_squared_distances(
    points_a: torch.Tensor, # (4,3)
    points_b: torch.Tensor, # (5,3)
) -> torch.Tensor:
    points_a = points_a.unsqueeze(1)
    points_b = points_b.unsqueeze(0)
    temp = (points_a - points_b)
    assert(temp.shape == (points_a.shape[0],points_b.shape[1],points_a.shape[2]))

    temp = temp * temp
    dist = temp.sum(2)
    return dist

slow = pairwise_squared_distances_slow(points_a, points_b)
fast = pairwise_squared_distances(points_a, points_b)

print(slow)
print(fast)

assert slow.shape == fast.shape == (4, 5) 
assert torch.allclose(slow, fast, atol=1e-6)

self_distances = pairwise_squared_distances(points_a, points_a)
print(self_distances)

tensor([[-0.1468,  0.7861,  0.9468],
        [-1.1143,  1.6908, -0.8948],
        [-0.3556,  1.2324,  0.1382],
        [-1.6822,  0.3177,  0.1328]])
tensor([[ 0.1373,  0.2405,  1.3955],
        [ 1.3470,  2.4382,  0.2028],
        [ 2.4505,  2.0256,  1.7792],
        [-0.9179, -0.4578, -0.7245],
        [ 1.2799, -0.9941,  1.8150]])
tensor([[ 0.5797,  5.5144,  8.9752,  4.9353,  5.9582],
        [ 8.9153,  7.8217, 19.9707,  4.6841, 20.2838],
        [ 2.8075,  4.3572, 11.1966,  3.9171, 10.4436],
        [ 4.9109, 13.6777, 22.7070,  1.9204, 13.3243]])
tensor([[ 0.5797,  5.5144,  8.9752,  4.9353,  5.9582],
        [ 8.9153,  7.8217, 19.9707,  4.6841, 20.2838],
        [ 2.8075,  4.3572, 11.1966,  3.9171, 10.4436],
        [ 4.9109, 13.6777, 22.7070,  1.9204, 13.3243]])
tensor([[0.0000, 5.1462, 0.8967, 3.2395],
        [5.1462, 0.0000, 1.8529, 3.2639],
        [0.8967, 1.8529, 0.0000, 2.5965],
        [3.2395, 3.2639, 2.5965, 0.0000]])


# Short little write-up

| Operand | Original shape | After insert axis |
| --- | --- | --- |
| `points_a` | `(N,D)` | `(N,1,D)` |
| `points_b` | `(M,D)` | `(1,M,D)` |

Broadcasting basically "expands" the missing dimension from either side, so for point_a[0,0,2], they will repeatedly be multiplied (or any element-wise operation) with points_b[0,x,2] along dimension 1.

In [13]:
# Devices

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda:0")
    else:
        return torch.device("cpu")

device = get_device()
print(device)

cuda:0


In [14]:
# Moving tensor to cuda

cpu_tensor = torch.randn(3, 4)
moved_tensor = cpu_tensor.to(device)

print(type(moved_tensor.device))
print(type(device))

assert cpu_tensor.device.type == "cpu"
assert moved_tensor.device == device

<class 'torch.device'>
<class 'torch.device'>


In [15]:
try:
    tensor1 = torch.randn(3,4)
    tensor2 = tensor1.to(get_device())
    tensor1 = tensor1 + tensor2
except RuntimeError as exc:
    print(type(exc).__name__)
    print(str(exc).splitlines()[0])

RuntimeError
Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!


In [16]:
try:
    tensor1 = torch.randn(3,4).to(get_device())
    tensor2 = torch.randn(3,4).to(get_device())
    result = tensor1 + tensor2
except RuntimeError as exc:
    print(type(exc).__name__)
    print(str(exc).splitlines()[0])

assert result.device == device
assert result.shape == (3,4)
assert torch.isfinite(result).all()

gpu_distances = pairwise_squared_distances(points_a.to(device), points_b.to(device))

assert gpu_distances.device == device
assert gpu_distances.shape == (points_a.shape[0], points_b.shape[0])
assert torch.allclose(gpu_distances.to(torch.device("cpu")), fast, atol=1e-6)